In [1]:
import os
import random
import tempfile
import shutil
from PIL import Image
import torch
import piq
import numpy as np
import pandas as pd


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
psnr_output_csv = "/data/liangz2/image_quality/psnr_scores.csv"


def load_image_as_tensor(path):
    """
    Load an RGB image from `path` into a float32 torch tensor in [0,1], shape (1, 3, H, W).
    """
    img = Image.open(path).convert("RGB")
    arr = np.array(img, dtype=np.float32) / 255.0     # shape (H, W, 3), values in [0,1]
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)  # (1, 3, H, W)
    return tensor

In [2]:
import os

def get_image_files(img_dir):
    image_files = []
    for subdir, _, files in os.walk(img_dir):
        for file in files:
            if file.endswith(('.png', '.jpg', '.jpeg')):
                # Get the absolute path and add it to the list
                image_files.append(os.path.abspath(os.path.join(subdir, file)))
    return image_files



real_img_dir_1 = '/data/liangz2/image_quality/real_images/mild'
real_img_dir_2 = '/data/liangz2/image_quality/real_images/severe'
sd21_i2i_mild_dir = '/data/liangz2/image_quality/fake_images/sd21/sd21_i2i_mild'
sd21_i2i_severe_dir = '/data/liangz2/image_quality/fake_images/sd21/sd21_i2i_severe'
sd21_t2i_mild_dir = '/data/liangz2/image_quality/fake_images/sd21/sd21_t2i_mild'
sd21_t2i_severe_dir = '/data/liangz2/image_quality/fake_images/sd21/sd21_t2i_severe'
sd35_i2i_mild_dir = '/data/liangz2/image_quality/fake_images/sd35/sd35_i2i_mild'
sd35_i2i_severe_dir = '/data/liangz2/image_quality/fake_images/sd35/sd35_i2i_severe'
sd35_t2i_mild_dir = '/data/liangz2/image_quality/fake_images/sd35/sd35_t2i_mild'
sd35_t2i_severe_dir = '/data/liangz2/image_quality/fake_images/sd35/sd35_t2i_severe'
stylegan2_mild_dir = '/data/liangz2/image_quality/fake_images/stylegan2/mild'
stylegan2_severe_dir = '/data/liangz2/image_quality/fake_images/stylegan2/severe'
stylegan3_mild_dir = '/data/liangz2/image_quality/fake_images/stylegan3/mild'
stylegan3_severe_dir = '/data/liangz2/image_quality/fake_images/stylegan3/severe'

real_mild_images = get_image_files(real_img_dir_1)
real_severe_images = get_image_files(real_img_dir_2)
sd21_i2i_mild_images = get_image_files(sd21_i2i_mild_dir)
sd21_i2i_severe_images = get_image_files(sd21_i2i_severe_dir)
sd21_t2i_mild_images = get_image_files(sd21_t2i_mild_dir)
sd21_t2i_severe_images = get_image_files(sd21_t2i_severe_dir)
sd35_i2i_mild_images = get_image_files(sd35_i2i_mild_dir)
sd35_i2i_severe_images = get_image_files(sd35_i2i_severe_dir)
sd35_t2i_mild_images = get_image_files(sd35_t2i_mild_dir)
sd35_t2i_severe_images = get_image_files(sd35_t2i_severe_dir)
stylegan2_mild_images = get_image_files(stylegan2_mild_dir)
stylegan2_severe_images = get_image_files(stylegan2_severe_dir)
stylegan3_mild_images = get_image_files(stylegan3_mild_dir)
stylegan3_severe_images = get_image_files(stylegan3_severe_dir)

print(f"total real mild images: {len(real_mild_images)}")
print(f"total real severe images: {len(real_severe_images)}")
print(f"total sd21 i2i mild images: {len(sd21_i2i_mild_images)}")
print(f"total sd21 i2i severe images: {len(sd21_i2i_severe_images)}")
print(f"total sd21 t2i mild images: {len(sd21_t2i_mild_images)}")
print(f"total sd21 t2i severe images: {len(sd21_t2i_severe_images)}")
print(f"total sd35 i2i mild images: {len(sd35_i2i_mild_images)}")
print(f"total sd35 i2i severe images: {len(sd35_i2i_severe_images)}")
print(f"total sd35 t2i mild images: {len(sd35_t2i_mild_images)}")
print(f"total sd35 t2i severe images: {len(sd35_t2i_severe_images)}")
print(f"total stylegan2 mild images: {len(stylegan2_mild_images)}")
print(f"total stylegan2 severe images: {len(stylegan2_severe_images)}")
print(f"total stylegan3 mild images: {len(stylegan3_mild_images)}")
print(f"total stylegan3 severe images: {len(stylegan3_severe_images)}")


def random_sample(image_set, sample_size=200):
    """
    Randomly sample a specified number of images from a list.
    """

    images = image_set.copy()
    if len(images) < sample_size:
        raise ValueError(f"Not enough images to sample {sample_size}. Available: {len(images)}")
    
    # Ensure reproducibility
    random.seed(42)
    random.shuffle(images)
    partitions = [images[i*200:(i+1)*200] for i in range(5)]

    # Example: access the first partition
    partition1 = partitions[0]
    partition2 = partitions[1]
    partition3 = partitions[2]
    partition4 = partitions[3]
    partition5 = partitions[4]
    return partition1, partition2, partition3, partition4, partition5

total real mild images: 200
total real severe images: 200
total sd21 i2i mild images: 1000
total sd21 i2i severe images: 1000
total sd21 t2i mild images: 1000
total sd21 t2i severe images: 1000
total sd35 i2i mild images: 1000
total sd35 i2i severe images: 1000
total sd35 t2i mild images: 1000
total sd35 t2i severe images: 1000
total stylegan2 mild images: 1000
total stylegan2 severe images: 1000
total stylegan3 mild images: 1000
total stylegan3 severe images: 1000


In [3]:
syn_images = {'sd21_i2i_mild':sd21_i2i_mild_images, 'sd21_i2i_severe':sd21_i2i_severe_images,
             'sd21_t2i_mild':sd21_t2i_mild_images, 'sd21_t2i_severe':sd21_t2i_severe_images,
             'sd35_i2i_mild':sd35_i2i_mild_images, 'sd35_i2i_severe':sd35_i2i_severe_images,
             'sd35_t2i_mild':sd35_t2i_mild_images, 'sd35_t2i_severe':sd35_t2i_severe_images,
             'stylegan2_mild':stylegan2_mild_images, 'stylegan2_severe':stylegan2_severe_images,
             'stylegan3_mild':stylegan3_mild_images, 'stylegan3_severe':stylegan3_severe_images}
real_images = {'mild': real_img_dir_1, 'severe': real_img_dir_2}

In [4]:
# ------------------------------
# Compute PSNR scores
# ------------------------------
psnr_score_dict = {}

for key, images in syn_images.items():
    print(f"======== processing {key} images... ========")
    img_type = key.split('_')[-1]  # 'mild' or 'severe'
    if img_type == 'mild':
        real_images = real_mild_images
        print(f"======== Using real mild images from reference ========")
    else:
        real_images = real_severe_images
        print(f"======== Using real severe images as reference ========")

    # Partition the synthetic images into five subsets
    sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5 = random_sample(images)
    partitions = [sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5]

    psnr_scores = []
    for idx, partition in enumerate(partitions, 1):
        print(f"Computing PSNR for partition {idx} with {len(partition)} images...")
        partition_psnrs = []

        for syn_path, real_path in zip(partition, real_images):
            # Load synthetic and real images as tensors, then move to DEVICE
            # Load synthetic and real images as tensors, resize to 224x224, then move to DEVICE
            syn_tensor  = load_image_as_tensor(syn_path)
            real_tensor = load_image_as_tensor(real_path)

            # Resize to 224x224
            syn_tensor = torch.nn.functional.interpolate(syn_tensor, size=(256, 256), mode='bilinear', align_corners=False)
            real_tensor = torch.nn.functional.interpolate(real_tensor, size=(256, 256), mode='bilinear', align_corners=False)

            syn_tensor = syn_tensor.to(DEVICE)
            real_tensor = real_tensor.to(DEVICE)

            # Compute PSNR (data_range=1.0 since tensors are in [0,1])
            with torch.no_grad():
                psnr_val = piq.psnr(syn_tensor, real_tensor, data_range=1.0).item()
            partition_psnrs.append(psnr_val)

        if partition_psnrs:
            avg_psnr = sum(partition_psnrs) / len(partition_psnrs)
        else:
            avg_psnr = float('nan')
        psnr_scores.append(avg_psnr)
        print(f"  Partition {idx} average PSNR: {avg_psnr:.2f} dB")

    psnr_score_dict[key] = psnr_scores

======== processing sd21_i2i_mild images... ========
======== Using real mild images from reference ========
Computing PSNR for partition 1 with 200 images...
  Partition 1 average PSNR: 12.48 dB
Computing PSNR for partition 2 with 200 images...
  Partition 2 average PSNR: 12.23 dB
Computing PSNR for partition 3 with 200 images...
  Partition 3 average PSNR: 12.51 dB
Computing PSNR for partition 4 with 200 images...
  Partition 4 average PSNR: 12.45 dB
Computing PSNR for partition 5 with 200 images...
  Partition 5 average PSNR: 12.36 dB
======== processing sd21_i2i_severe images... ========
======== Using real severe images as reference ========
Computing PSNR for partition 1 with 200 images...
  Partition 1 average PSNR: 12.44 dB
Computing PSNR for partition 2 with 200 images...
  Partition 2 average PSNR: 11.98 dB
Computing PSNR for partition 3 with 200 images...
  Partition 3 average PSNR: 12.28 dB
Computing PSNR for partition 4 with 200 images...
  Partition 4 average PSNR: 12.19 

In [5]:
# ------------------------------
# Save results to CSV
# ------------------------------
psnr_df = pd.DataFrame({k: pd.Series(v) for k, v in psnr_score_dict.items()})
psnr_df.to_csv(psnr_output_csv, index=False)
print(f"PSNR scores computed and saved to: {psnr_output_csv}")
print("PSNR scores:")
print(psnr_df)

PSNR scores computed and saved to: /data/liangz2/image_quality/psnr_scores.csv
PSNR scores:
   sd21_i2i_mild  sd21_i2i_severe  sd21_t2i_mild  sd21_t2i_severe  \
0      12.476814        12.442620      12.109314        11.190639   
1      12.226238        11.975750      12.176993        11.209939   
2      12.509702        12.281396      12.141956        11.217306   
3      12.453844        12.189612      12.118471        11.321443   
4      12.361987        12.255794      12.083581        11.150553   

   sd35_i2i_mild  sd35_i2i_severe  sd35_t2i_mild  sd35_t2i_severe  \
0      12.878302        12.590751      12.707466        11.776718   
1      12.663573        12.255552      12.720198        11.760874   
2      12.936819        12.439085      12.736746        11.803282   
3      12.843564        12.376879      12.740547        11.744390   
4      12.812993        12.469738      12.700316        11.763901   

   stylegan2_mild  stylegan2_severe  stylegan3_mild  stylegan3_severe  
0     

In [6]:
from scipy.stats import ttest_ind

# Perform pairwise t-tests between all columns in psnr_df
columns = psnr_df.columns
results = []
root_path = "/data/liangz2/image_quality"

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1 = columns[i]
        col2 = columns[j]
        # Drop NaN values for fair comparison
        data1 = psnr_df[col1].dropna()
        data2 = psnr_df[col2].dropna()
        # Perform independent t-test
        t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
        results.append({
            'Group1': col1,
            'Group2': col2,
            't_stat': t_stat,
            'p_value': p_val
        })
        print(f"{col1} vs {col2}: t-stat={t_stat:.4f}, p-value={p_val:.4e}")

# Save results to CSV
ttest_df = pd.DataFrame(results)
save_path = os.path.join(root_path, "psnr_pairwise_ttest_results.csv")
ttest_df.to_csv(save_path, index=False)
print(f"Pairwise t-test results saved to {save_path}")

# Compute mean and std for each column
psnr_stats = psnr_df.agg(['mean', 'std']).transpose().reset_index()
psnr_stats.columns = ['Group', 'Mean', 'Std']

# Save to CSV
stats_save_path = os.path.join(root_path, "psnr_mean_std.csv")
psnr_stats.to_csv(stats_save_path, index=False)
print(f"Mean and std of PSNR scores saved to {stats_save_path}")
print(psnr_stats)

sd21_i2i_mild vs sd21_i2i_severe: t-stat=1.9330, p-value=9.4394e-02
sd21_i2i_mild vs sd21_t2i_mild: t-stat=5.2248, p-value=3.9288e-03
sd21_i2i_mild vs sd21_t2i_severe: t-stat=20.3140, p-value=6.1069e-07
sd21_i2i_mild vs sd35_i2i_mild: t-stat=-6.1399, p-value=2.9109e-04
sd21_i2i_mild vs sd35_i2i_severe: t-stat=-0.2752, p-value=7.9019e-01
sd21_i2i_mild vs sd35_t2i_mild: t-stat=-6.0941, p-value=3.1497e-03
sd21_i2i_mild vs sd35_t2i_severe: t-stat=12.2107, p-value=1.6985e-04
sd21_i2i_mild vs stylegan2_mild: t-stat=-19.1665, p-value=1.3254e-07
sd21_i2i_mild vs stylegan2_severe: t-stat=-16.9117, p-value=9.4639e-07
sd21_i2i_mild vs stylegan3_mild: t-stat=-16.4134, p-value=2.7133e-07
sd21_i2i_mild vs stylegan3_severe: t-stat=-28.9021, p-value=3.3202e-08
sd21_i2i_severe vs sd21_t2i_mild: t-stat=1.3306, p-value=2.4883e-01
sd21_i2i_severe vs sd21_t2i_severe: t-stat=12.5000, p-value=5.1005e-05
sd21_i2i_severe vs sd35_i2i_mild: t-stat=-6.7570, p-value=3.4645e-04
sd21_i2i_severe vs sd35_i2i_severe: t

In [4]:
# ------------------------------
# Compute LPIPS scores (Perceptual Similarity)
# ------------------------------
import torchvision.transforms as T
from piq import LPIPS

# Load LPIPS model and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lpips_model = LPIPS().to(device)
lpips_score_dict = {}

# Load images and convert to tensors
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    lambda x: x * 2 - 1  # Normalize to [-1, 1] for LPIPS
])

def get_lpips(real_path, fake_path):
    real_image = Image.open(real_path).convert('RGB')
    fake_image = Image.open(fake_path).convert('RGB')
    img1 = transform(real_image).unsqueeze(0)  # shape: (1, 3, H, W)
    img2 = transform(fake_image).unsqueeze(0)
    # Normalize to [-1, 1] and move to GPU
    img1 = (img1 * 2 - 1).to(device)
    img2 = (img2 * 2 - 1).to(device)
    # Compute LPIPS
    with torch.no_grad():
        score = lpips_model(img1, img2)
    return score.mean().item()


for key, images in syn_images.items():
    print(f"======== processing {key} images... ========")
    img_type = key.split('_')[-1]  # 'mild' or 'severe'
    if img_type == 'mild':
        real_images = real_mild_images
        print(f"======== Using real mild images from reference ========")
    else:
        real_images = real_severe_images
        print(f"======== Using real severe images as reference ========")

    # Partition the synthetic images into five subsets
    sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5 = random_sample(images)
    partitions = [sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5]

    lpips_scores = []
    for idx, partition in enumerate(partitions, 1):
        print(f"Computing LPIPS for partition {idx} with {len(partition)} images...")
        partition_lpips = []

        for syn_path, real_path in zip(partition, real_images):
            # Load synthetic and real images as tensors, then move to DEVICE
            # Load synthetic and real images as tensors, resize to 224x224, then move to DEVICE
            single_lpips = get_lpips(real_path, syn_path)
            partition_lpips.append(single_lpips)

        if partition_lpips:
            avg_lpips = sum(partition_lpips) / len(partition_lpips)
        else:
            avg_lpips = float('nan')
        lpips_scores.append(avg_lpips)
        print(f"  Partition {idx} average LPIPS: {avg_lpips:.2f}")

    lpips_score_dict[key] = lpips_scores

/data/liangz2/imgeval/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/data/liangz2/imgeval/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


======== processing sd21_i2i_mild images... ========
======== Using real mild images from reference ========
Computing LPIPS for partition 1 with 200 images...
  Partition 1 average LPIPS: 0.47
Computing LPIPS for partition 2 with 200 images...
  Partition 2 average LPIPS: 0.48
Computing LPIPS for partition 3 with 200 images...
  Partition 3 average LPIPS: 0.48
Computing LPIPS for partition 4 with 200 images...
  Partition 4 average LPIPS: 0.48
Computing LPIPS for partition 5 with 200 images...
  Partition 5 average LPIPS: 0.48
======== processing sd21_i2i_severe images... ========
======== Using real severe images as reference ========
Computing LPIPS for partition 1 with 200 images...
  Partition 1 average LPIPS: 0.49
Computing LPIPS for partition 2 with 200 images...
  Partition 2 average LPIPS: 0.51
Computing LPIPS for partition 3 with 200 images...
  Partition 3 average LPIPS: 0.50
Computing LPIPS for partition 4 with 200 images...
  Partition 4 average LPIPS: 0.51
Computing LPIPS

In [7]:
# ------------------------------
# Save results to CSV
# ------------------------------
lpips_output_csv = "/data/liangz2/image_quality/lpips_scores.csv"
lpips_df = pd.DataFrame({k: pd.Series(v) for k, v in lpips_score_dict.items()})

lpips_df.to_csv(lpips_output_csv, index=False)
print(f"LPIPS scores computed and saved to: {lpips_output_csv}")
print("LPIPS scores:")
print(lpips_df)

LPIPS scores computed and saved to: /data/liangz2/image_quality/lpips_scores.csv
LPIPS scores:
   sd21_i2i_mild  sd21_i2i_severe  sd21_t2i_mild  sd21_t2i_severe  \
0       0.473710         0.490171       0.491343         0.525200   
1       0.481064         0.505588       0.490028         0.526653   
2       0.475473         0.496721       0.492634         0.526280   
3       0.482875         0.505014       0.492597         0.524322   
4       0.478981         0.499867       0.492903         0.527146   

   sd35_i2i_mild  sd35_i2i_severe  sd35_t2i_mild  sd35_t2i_severe  \
0       0.459142         0.483702       0.463875         0.501677   
1       0.467118         0.497206       0.464278         0.500681   
2       0.461193         0.489561       0.463704         0.500595   
3       0.468270         0.495410       0.462959         0.502300   
4       0.464408         0.492256       0.465170         0.501722   

   stylegan2_mild  stylegan2_severe  stylegan3_mild  stylegan3_severe  
0  

In [9]:
# Perform pairwise t-tests between all columns in lpips_df
from scipy.stats import ttest_ind

columns = lpips_df.columns
results = []
root_path = "/data/liangz2/image_quality"

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1 = columns[i]
        col2 = columns[j]
        # Drop NaN values for fair comparison
        data1 = lpips_df[col1].dropna()
        data2 = lpips_df[col2].dropna()
        # Perform independent t-test
        t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
        results.append({
            'Group1': col1,
            'Group2': col2,
            't_stat': t_stat,
            'p_value': p_val
        })
        print(f"{col1} vs {col2}: t-stat={t_stat:.4f}, p-value={p_val:.4e}")

# Save results to CSV
ttest_df = pd.DataFrame(results)
save_path = os.path.join(root_path, "lpips_ttest_results.csv")
ttest_df.to_csv(save_path, index=False)
print(f"Pairwise t-test results saved to {save_path}")

# Compute mean and std for each column
lpips_stats = lpips_df.agg(['mean', 'std']).transpose().reset_index()
lpips_stats.columns = ['Group', 'Mean', 'Std']

# Save to CSV
stats_save_path = os.path.join(root_path, "lpips_mean_std.csv")
lpips_stats.to_csv(stats_save_path, index=False)
print(f"Mean and std of LPIPS scores saved to {stats_save_path}")
print(lpips_stats)

sd21_i2i_mild vs sd21_i2i_severe: t-stat=-6.3421, p-value=5.1289e-04
sd21_i2i_mild vs sd21_t2i_mild: t-stat=-7.5440, p-value=7.7641e-04
sd21_i2i_mild vs sd21_t2i_severe: t-stat=-26.7088, p-value=2.4875e-06
sd21_i2i_mild vs sd35_i2i_mild: t-stat=5.9357, p-value=3.4775e-04
sd21_i2i_mild vs sd35_i2i_severe: t-stat=-4.5176, p-value=2.5067e-03
sd21_i2i_mild vs sd35_t2i_mild: t-stat=8.2828, p-value=7.9071e-04
sd21_i2i_mild vs sd35_t2i_severe: t-stat=-13.2456, p-value=1.2028e-04
sd21_i2i_mild vs stylegan2_mild: t-stat=25.5989, p-value=1.9716e-06
sd21_i2i_mild vs stylegan2_severe: t-stat=20.6602, p-value=3.4607e-08
sd21_i2i_mild vs stylegan3_mild: t-stat=27.9302, p-value=1.1737e-07
sd21_i2i_mild vs stylegan3_severe: t-stat=26.2212, p-value=4.9579e-09
sd21_i2i_severe vs sd21_t2i_mild: t-stat=2.6109, p-value=5.5378e-02
sd21_i2i_severe vs sd21_t2i_severe: t-stat=-9.1369, p-value=5.9135e-04
sd21_i2i_severe vs sd35_i2i_mild: t-stat=10.6404, p-value=2.1670e-05
sd21_i2i_severe vs sd35_i2i_severe: t-s

In [5]:
# ------------------------------
# Compute DISTS scores (Deep Image Structure and Texture Similarity)
# ------------------------------
import torchvision.transforms as T
from piq import DISTS

# Load LPIPS model and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dists_metric = DISTS()
dists_score_dict = {}

# Load images and convert to tensors
transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    lambda x: x * 2 - 1  # Normalize to [-1, 1] for LPIPS
])

def get_dists(real_path, fake_path):
    real_image = Image.open(real_path).convert('RGB')
    fake_image = Image.open(fake_path).convert('RGB')
    img1 = transform(real_image).unsqueeze(0)  # shape: (1, 3, H, W)
    img2 = transform(fake_image).unsqueeze(0)
    # Normalize to [-1, 1] and move to GPU
    img1, img2 = img1.to(device), img2.to(device)
    # Compute LPIPS
    with torch.no_grad():
        score = dists_metric(img1, img2)
    return score.item()

Downloading: "https://github.com/photosynthesis-team/piq/releases/download/v0.4.1/dists_weights.pt" to /home/liangz2/.cache/torch/hub/checkpoints/dists_weights.pt


/data/liangz2/imgeval/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/data/liangz2/imgeval/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
for key, images in syn_images.items():
    print(f"======== processing {key} images... ========")
    img_type = key.split('_')[-1]  # 'mild' or 'severe'
    if img_type == 'mild':
        real_images = real_mild_images
        print(f"======== Using real mild images from reference ========")
    else:
        real_images = real_severe_images
        print(f"======== Using real severe images as reference ========")

    # Partition the synthetic images into five subsets
    sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5 = random_sample(images)
    partitions = [sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5]

    dists_scores = []
    for idx, partition in enumerate(partitions, 1):
        print(f"Computing DISTS for partition {idx} with {len(partition)} images...")
        partition_dists = []

        for syn_path, real_path in zip(partition, real_images):
            # Load synthetic and real images as tensors, then move to DEVICE
            # Load synthetic and real images as tensors, resize to 224x224, then move to DEVICE
            single_dists = get_dists(real_path, syn_path)
            partition_dists.append(single_dists)

        if partition_dists:
            avg_dists = sum(partition_dists) / len(partition_dists)
        else:
            avg_dists = float('nan')
        dists_scores.append(avg_dists)
        print(f"  Partition {idx} average DISTS: {avg_dists:.2f}")

    dists_score_dict[key] = dists_scores

======== processing sd21_i2i_mild images... ========
======== Using real mild images from reference ========
Computing DISTS for partition 1 with 200 images...
  Partition 1 average DISTS: 0.37
Computing DISTS for partition 2 with 200 images...
  Partition 2 average DISTS: 0.38
Computing DISTS for partition 3 with 200 images...
  Partition 3 average DISTS: 0.37
Computing DISTS for partition 4 with 200 images...
  Partition 4 average DISTS: 0.37
Computing DISTS for partition 5 with 200 images...
  Partition 5 average DISTS: 0.37
======== processing sd21_i2i_severe images... ========
======== Using real severe images as reference ========
Computing DISTS for partition 1 with 200 images...
  Partition 1 average DISTS: 0.42
Computing DISTS for partition 2 with 200 images...
  Partition 2 average DISTS: 0.43
Computing DISTS for partition 3 with 200 images...
  Partition 3 average DISTS: 0.42
Computing DISTS for partition 4 with 200 images...
  Partition 4 average DISTS: 0.42
Computing DISTS

In [8]:
# ------------------------------
# Save results to CSV
# ------------------------------
dists_output_csv = "/data/liangz2/image_quality/dists_scores.csv"
dists_df = pd.DataFrame({k: pd.Series(v) for k, v in dists_score_dict.items()})

dists_df.to_csv(dists_output_csv, index=False)
print(f"DISTS scores computed and saved to: {dists_output_csv}")
print("DISTS scores:")
print(dists_df)

DISTS scores computed and saved to: /data/liangz2/image_quality/dists_scores.csv
DISTS scores:
   sd21_i2i_mild  sd21_i2i_severe  sd21_t2i_mild  sd21_t2i_severe  \
0       0.374614         0.418781       0.385567         0.460775   
1       0.378044         0.425214       0.382307         0.465441   
2       0.371886         0.415106       0.383600         0.460787   
3       0.372305         0.422858       0.386133         0.455544   
4       0.374709         0.417220       0.383342         0.465462   

   sd35_i2i_mild  sd35_i2i_severe  sd35_t2i_mild  sd35_t2i_severe  \
0       0.370504         0.434636       0.364962         0.443822   
1       0.372755         0.433702       0.366641         0.442658   
2       0.369081         0.429146       0.364026         0.444013   
3       0.367979         0.435958       0.363428         0.445619   
4       0.369880         0.432472       0.365554         0.444424   

   stylegan2_mild  stylegan2_severe  stylegan3_mild  stylegan3_severe  
0  

In [9]:
# Perform pairwise t-tests between all columns in lpips_df
from scipy.stats import ttest_ind

columns = dists_df.columns
results = []
root_path = "/data/liangz2/image_quality"

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1 = columns[i]
        col2 = columns[j]
        # Drop NaN values for fair comparison
        data1 = dists_df[col1].dropna()
        data2 = dists_df[col2].dropna()
        # Perform independent t-test
        t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
        results.append({
            'Group1': col1,
            'Group2': col2,
            't_stat': t_stat,
            'p_value': p_val
        })
        print(f"{col1} vs {col2}: t-stat={t_stat:.4f}, p-value={p_val:.4e}")

# Save results to CSV
ttest_df = pd.DataFrame(results)
save_path = os.path.join(root_path, "dists_ttest_results.csv")
ttest_df.to_csv(save_path, index=False)
print(f"Pairwise t-test results saved to {save_path}")

# Compute mean and std for each column
dists_stats = dists_df.agg(['mean', 'std']).transpose().reset_index()
dists_stats.columns = ['Group', 'Mean', 'Std']

# Save to CSV
stats_save_path = os.path.join(root_path, "dists_mean_std.csv")
dists_stats.to_csv(stats_save_path, index=False)
print(f"Mean and std of DISTS scores saved to {stats_save_path}")
print(dists_stats)

sd21_i2i_mild vs sd21_i2i_severe: t-stat=-21.1721, p-value=3.0198e-07
sd21_i2i_mild vs sd21_t2i_mild: t-stat=-7.5348, p-value=1.4454e-04
sd21_i2i_mild vs sd21_t2i_severe: t-stat=-40.7500, p-value=4.1903e-09
sd21_i2i_mild vs sd35_i2i_mild: t-stat=3.1463, p-value=1.5342e-02
sd21_i2i_mild vs sd35_i2i_severe: t-stat=-36.8767, p-value=3.3742e-10
sd21_i2i_mild vs sd35_t2i_mild: t-stat=7.6055, p-value=2.7270e-04
sd21_i2i_mild vs sd35_t2i_severe: t-stat=-58.2979, p-value=7.5123e-09
sd21_i2i_mild vs stylegan2_mild: t-stat=-7.6694, p-value=9.5910e-04
sd21_i2i_mild vs stylegan2_severe: t-stat=-13.9314, p-value=3.4780e-05
sd21_i2i_mild vs stylegan3_mild: t-stat=-6.4847, p-value=1.4565e-03
sd21_i2i_mild vs stylegan3_severe: t-stat=-13.6205, p-value=3.0753e-05
sd21_i2i_severe vs sd21_t2i_mild: t-stat=17.9744, p-value=7.2798e-06
sd21_i2i_severe vs sd21_t2i_severe: t-stat=-16.0132, p-value=2.3199e-07
sd21_i2i_severe vs sd35_i2i_mild: t-stat=24.7200, p-value=8.3906e-07
sd21_i2i_severe vs sd35_i2i_sever

In [15]:
# ------------------------------
# Compute BRISQUE scores (Blind/Referenceless Image Spatial Quality Evaluator)
# ------------------------------
import torchvision.transforms as T
from piq import brisque

# Load LPIPS model and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
brisque_score_dict = {}

# Load images and convert to tensors
brisque_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    lambda x: x.unsqueeze(0)  # Add batch dimension
])

def get_brisque(image_path):
    img = Image.open(image_path).convert("RGB")
    img_tensor = brisque_transform(img)
    score = brisque(img_tensor, data_range=1.)
    return score.item()

In [16]:
for key, images in syn_images.items():
    # Partition the synthetic images into five subsets
    sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5 = random_sample(images)
    partitions = [sub_sample1, sub_sample2, sub_sample3, sub_sample4, sub_sample5]

    brisque_scores = []
    for idx, partition in enumerate(partitions, 1):
        print(f"Computing BRISQUE for partition {idx} with {len(partition)} images...")
        partition_brisque = []

        for syn_path in partition:
            single_brisque = get_brisque(syn_path)
            partition_brisque.append(single_brisque)

        if partition_brisque:
            avg_brisque = sum(partition_brisque) / len(partition_brisque)
        else:
            avg_brisque = float('nan')
        brisque_scores.append(avg_brisque)
        print(f"  Partition {idx} average BRISQUE: {avg_brisque:.2f}")

    brisque_score_dict[key] = brisque_scores

Computing BRISQUE for partition 1 with 200 images...
Downloading: "https://github.com/photosynthesis-team/piq/releases/download/v0.4.0/brisque_svm_weights.pt" to /home/liangz2/.cache/torch/hub/checkpoints/brisque_svm_weights.pt


100%|██████████| 112k/112k [00:00<00:00, 18.3MB/s]


  Partition 1 average BRISQUE: 21.53
Computing BRISQUE for partition 2 with 200 images...
  Partition 2 average BRISQUE: 22.86
Computing BRISQUE for partition 3 with 200 images...
  Partition 3 average BRISQUE: 22.25
Computing BRISQUE for partition 4 with 200 images...
  Partition 4 average BRISQUE: 22.12
Computing BRISQUE for partition 5 with 200 images...
  Partition 5 average BRISQUE: 22.05
Computing BRISQUE for partition 1 with 200 images...
  Partition 1 average BRISQUE: 20.17
Computing BRISQUE for partition 2 with 200 images...
  Partition 2 average BRISQUE: 21.56
Computing BRISQUE for partition 3 with 200 images...
  Partition 3 average BRISQUE: 21.31
Computing BRISQUE for partition 4 with 200 images...
  Partition 4 average BRISQUE: 20.85
Computing BRISQUE for partition 5 with 200 images...
  Partition 5 average BRISQUE: 20.65
Computing BRISQUE for partition 1 with 200 images...
  Partition 1 average BRISQUE: 23.82
Computing BRISQUE for partition 2 with 200 images...
  Partitio

In [17]:
# ------------------------------
# Save results to CSV
# ------------------------------
brisque_output_csv = "/data/liangz2/image_quality/brisque_scores.csv"
brisque_df = pd.DataFrame({k: pd.Series(v) for k, v in brisque_score_dict.items()})

brisque_df.to_csv(brisque_output_csv, index=False)
print(f"BRISQUE scores computed and saved to: {brisque_output_csv}")
print("BRISQUE scores:")
print(brisque_df)

BRISQUE scores computed and saved to: /data/liangz2/image_quality/brisque_scores.csv
BRISQUE scores:
   sd21_i2i_mild  sd21_i2i_severe  sd21_t2i_mild  sd21_t2i_severe  \
0      21.528019        20.172233      23.823391        24.387451   
1      22.863571        21.558948      23.827755        24.475210   
2      22.250850        21.311931      23.651853        24.401118   
3      22.124384        20.849340      23.494286        24.384666   
4      22.054393        20.649880      23.765522        24.431132   

   sd35_i2i_mild  sd35_i2i_severe  sd35_t2i_mild  sd35_t2i_severe  \
0      26.726568        24.124030      23.527635        21.445728   
1      26.784975        24.196738      23.422745        20.786942   
2      26.545414        24.103855      23.326137        21.010289   
3      26.261484        24.395981      22.993080        21.340282   
4      26.706301        24.088664      22.828771        20.857058   

   stylegan2_mild  stylegan2_severe  stylegan3_mild  stylegan3_severe

In [19]:
# Perform pairwise t-tests between all columns in lpips_df
from scipy.stats import ttest_ind

columns = brisque_df.columns
results = []
root_path = "/data/liangz2/image_quality"

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1 = columns[i]
        col2 = columns[j]
        # Drop NaN values for fair comparison
        data1 = dists_df[col1].dropna()
        data2 = dists_df[col2].dropna()
        # Perform independent t-test
        t_stat, p_val = ttest_ind(data1, data2, equal_var=False)
        results.append({
            'Group1': col1,
            'Group2': col2,
            't_stat': t_stat,
            'p_value': p_val
        })
        print(f"{col1} vs {col2}: t-stat={t_stat:.4f}, p-value={p_val:.4e}")

# Save results to CSV
ttest_df = pd.DataFrame(results)
save_path = os.path.join(root_path, "brisque_ttest_results.csv")
ttest_df.to_csv(save_path, index=False)
print(f"Pairwise t-test results saved to {save_path}")

# Compute mean and std for each column
brisque_stats = brisque_df.agg(['mean', 'std']).transpose().reset_index()
brisque_stats.columns = ['Group', 'Mean', 'Std']

# Save to CSV
stats_save_path = os.path.join(root_path, "brisque_mean_std.csv")
brisque_stats.to_csv(stats_save_path, index=False)
print(f"Mean and std of BRISQUE scores saved to {stats_save_path}")
print(brisque_stats)

sd21_i2i_mild vs sd21_i2i_severe: t-stat=-21.1721, p-value=3.0198e-07
sd21_i2i_mild vs sd21_t2i_mild: t-stat=-7.5348, p-value=1.4454e-04
sd21_i2i_mild vs sd21_t2i_severe: t-stat=-40.7500, p-value=4.1903e-09
sd21_i2i_mild vs sd35_i2i_mild: t-stat=3.1463, p-value=1.5342e-02
sd21_i2i_mild vs sd35_i2i_severe: t-stat=-36.8767, p-value=3.3742e-10
sd21_i2i_mild vs sd35_t2i_mild: t-stat=7.6055, p-value=2.7270e-04
sd21_i2i_mild vs sd35_t2i_severe: t-stat=-58.2979, p-value=7.5123e-09
sd21_i2i_mild vs stylegan2_mild: t-stat=-7.6694, p-value=9.5910e-04
sd21_i2i_mild vs stylegan2_severe: t-stat=-13.9314, p-value=3.4780e-05
sd21_i2i_mild vs stylegan3_mild: t-stat=-6.4847, p-value=1.4565e-03
sd21_i2i_mild vs stylegan3_severe: t-stat=-13.6205, p-value=3.0753e-05
sd21_i2i_severe vs sd21_t2i_mild: t-stat=17.9744, p-value=7.2798e-06
sd21_i2i_severe vs sd21_t2i_severe: t-stat=-16.0132, p-value=2.3199e-07
sd21_i2i_severe vs sd35_i2i_mild: t-stat=24.7200, p-value=8.3906e-07
sd21_i2i_severe vs sd35_i2i_sever